# Shared infrastructure for the three model tracks

Run this **once, before anyone starts modelling**. It writes the artifacts that
the GNN, xLSTM and Chronos notebooks all import, so the three sets of numbers
are actually comparable.

It produces, into `dataset/shared/`:

| file | what it fixes |
|---|---|
| `split.json` | the frozen temporal split and the masking convention |
| `station_groups.json` | which pollutants each station can actually measure |
| `station_coords.csv` | lat/lon/elevation/type, for the GNN adjacency |
| `eval_windows.csv` | the exact forecast windows every track must score on |
| `bih_shared.py` | one implementation of loading, masking and metrics |
| `baseline_metrics.csv` | the bar all three model families have to clear |

The output folder is published at
<https://drive.google.com/drive/folders/1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE> - the other three notebooks
download it with `gdown.download_folder` and never rebuild it themselves.

Nothing here trains anything. Each track imports `bih_shared.py` and gets the
same split, the same mask and the same metric functions - not three
re-derivations that drift apart.

In [1]:
# ---- data source -------------------------------------------------------
# Where the artifacts this notebook produces are published. The other tracks
# download this folder instead of re-running the notebook.
SHARED_FOLDER_ID = "1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE"

FILE_IDS = {
    "bih_dataset/bih_hourly_clean.csv":                "1QyfzghimgyRhQ735k42JetDEoK5kfGZJ",
    # The station register is embedded further down, so this one is optional.
    "fhz_data/pollutants/meta podaci.xlsx":            "",
}

DRIVE_DATASET = "/content/drive/MyDrive/air_pollution_bih/dataset"
LOCAL_DATASET = "/home/sehy/Downloads/air_pollution_bih/dataset"

import os, sys, json, subprocess
import numpy as np
import pandas as pd


def _gdown():
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    return gdown


def fetch_from_drive(file_ids, dest="dataset"):
    gdown = None
    for rel, fid in file_ids.items():
        if not fid.strip():          # optional file, skipped
            continue
        out = os.path.join(dest, rel)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            print(f"have  {rel}")
            continue
        gdown = gdown or _gdown()
        os.makedirs(os.path.dirname(out), exist_ok=True)
        print(f"get   {rel}")
        gdown.download(id=fid, output=out, quiet=False)
        if not os.path.exists(out) or os.path.getsize(out) == 0:
            raise RuntimeError(
                f"download failed for {rel}. Check the ID and that the file is "
                f"shared as 'Anyone with the link'."
            )
    return dest


if any(v.strip() for v in FILE_IDS.values()):
    DATASET = fetch_from_drive(FILE_IDS)
elif os.path.isdir(LOCAL_DATASET):
    DATASET = LOCAL_DATASET
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET = DRIVE_DATASET

CLEAN  = os.path.join(DATASET, "bih_dataset", "bih_hourly_clean.csv")
META   = os.path.join(DATASET, "fhz_data", "pollutants", "meta podaci.xlsx")
SHARED = os.path.join(DATASET, "shared")
os.makedirs(SHARED, exist_ok=True)
print("\nclean :", CLEAN, "|", os.path.exists(CLEAN))
print("meta  :", META, "|", os.path.exists(META))
print("out   :", SHARED)

get   bih_dataset/bih_hourly_clean.csv


Downloading...
From (original): https://drive.google.com/uc?id=1QyfzghimgyRhQ735k42JetDEoK5kfGZJ
From (redirected): https://drive.google.com/uc?id=1QyfzghimgyRhQ735k42JetDEoK5kfGZJ&confirm=t&uuid=c1dbf8cb-a4fe-4138-a384-a71bbc1302c5
To: /content/dataset/bih_dataset/bih_hourly_clean.csv
100%|██████████| 167M/167M [00:01<00:00, 137MB/s]


clean : dataset/bih_dataset/bih_hourly_clean.csv | True
meta  : dataset/fhz_data/pollutants/meta podaci.xlsx | False
out   : dataset/shared


## Check the file is the right one

The clean export must carry a `_filled` flag for every pollutant. Without those
flags there is no way to tell a real measurement from a linear interpolation,
and every metric below would be partly scoring the interpolation. This cell
stops now rather than letting that happen silently.

In [2]:
POLLUTANTS = ["pm10", "pm25", "so2", "no2", "o3", "co"]
MET        = ["temperature", "humidity", "wind_speed", "wind_dir", "pressure",
              "wind_u", "wind_v"]

header = pd.read_csv(CLEAN, nrows=0).columns.tolist()
need   = (["source", "city", "station", "datetime", "year", "season"]
          + POLLUTANTS + [f"{p}_filled" for p in POLLUTANTS])
missing = [c for c in need if c not in header]

print(f"{len(header)} columns in {os.path.basename(CLEAN)}")
if missing:
    raise SystemExit(
        f"\nThis is not the current clean export - missing: {missing}\n\n"
        f"The correct file has 32 columns and is ~167 MB. Re-upload it (or\n"
        f"re-run build_dataset.ipynb) before continuing. Do not work around\n"
        f"this by assuming nothing was filled: gaps of up to 3 h were\n"
        f"interpolated, and scoring against them inflates every result."
    )
print("schema OK - all six pollutant fill flags present")

32 columns in bih_hourly_clean.csv
schema OK - all six pollutant fill flags present


In [3]:
usecols = (["source", "city", "station", "datetime", "year", "season"]
           + POLLUTANTS + [f"{p}_filled" for p in POLLUTANTS]
           + [c for c in MET if c in header])
df = pd.read_csv(CLEAN, usecols=usecols, parse_dates=["datetime"])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)

for p in POLLUTANTS:
    df[f"{p}_filled"] = df[f"{p}_filled"].fillna(False).astype(bool)

print(df.shape, "|", df.datetime.min(), "->", df.datetime.max())
print(df.station.nunique(), "stations,", df.city.nunique(), "cities")

(753823, 25) | 2021-01-01 00:00:00 -> 2024-12-31 23:00:00
23 stations, 12 cities


## 1. Freeze the split

Temporal, never random: a random split would put 2023-07-04 09:00 in train and
10:00 in validation, and any model would score brilliantly by copying the hour
next to it.

- **train** 2021-01-01 .. 2023-12-31 (three full years)
- **val**   2024-01-01 .. 2024-12-31 (one full year, all four seasons)

2025 is excluded entirely: FHZ stops after 2024, so a 2025 fold would be RHZ
stations only and not comparable across the network.

There is no separate test year. With one year of held-out data, calling it
*validation* is honest - every tuning decision the three tracks make will have
seen it. Say so in the paper rather than calling it a test set.

In [4]:
SPLIT = {
    "train": {"start": "2021-01-01 00:00", "end": "2023-12-31 23:00"},
    "val":   {"start": "2024-01-01 00:00", "end": "2024-12-31 23:00"},
    "excluded": {"2025": "FHZ has no 2025 data; an RHZ-only year is not comparable"},

    "horizon_hours": 24,
    "origin_stride_hours": 24,
    "origin_hour_of_day": 0,

    "masking": {
        "rule": "score only where value is not NaN AND <pollutant>_filled is False",
        "why": ("gaps of up to 3 h were linearly interpolated during dataset "
                "build; scoring against them measures our own interpolation, "
                "not the model"),
        "filled_allowed_in_input": True,
        "filled_allowed_in_target": False,
        "min_real_target_hours": 12,
        "min_real_context_hours_168": 1,
    },

    "metrics": ["mae", "rmse", "mase"],
    "metric_note": ("MAE and RMSE are per pollutant, in native units, and must "
                    "never be averaged across pollutants - CO is mg/m3 and the "
                    "rest are ug/m3. MASE is the only cross-pollutant summary."),
}

TRAIN_END = pd.Timestamp(SPLIT["train"]["end"])
VAL_START = pd.Timestamp(SPLIT["val"]["start"])
VAL_END   = pd.Timestamp(SPLIT["val"]["end"])
HORIZON   = SPLIT["horizon_hours"]

with open(os.path.join(SHARED, "split.json"), "w") as f:
    json.dump(SPLIT, f, indent=2)

tr = df[df.datetime <= TRAIN_END]
va = df[(df.datetime >= VAL_START) & (df.datetime <= VAL_END)]
print(f"train {len(tr):>7,} rows  {tr.datetime.min()} -> {tr.datetime.max()}")
print(f"val   {len(va):>7,} rows  {va.datetime.min()} -> {va.datetime.max()}")
print("\nwrote split.json")

train 560,597 rows  2021-01-01 00:00:00 -> 2023-12-31 23:00:00
val   193,226 rows  2024-01-01 00:00:00 -> 2024-12-31 23:00:00

wrote split.json


## 2. Freeze the station-pollutant partitions

Three states, and the difference matters:

- **never** - the instrument does not exist at that station. The column is
  empty in the source. This is not missing data and must not be imputed;
  the station is simply out of scope for that pollutant.
- **sparse** - the instrument exists but reports under half the time.
  Usable with care, and worth reporting separately rather than pooling.
- **ok** - at least half the hours are real measurements.

Derived once, here, from the data - so all three tracks scope their stations
identically instead of each picking their own threshold.

In [5]:
OK_THRESHOLD = 0.50

rows = []
for (src, city, st), g in df.groupby(["source", "city", "station"], sort=True):
    for p in POLLUTANTS:
        real = g[p].notna() & ~g[f"{p}_filled"]
        cov = real.mean()
        rows.append({
            "source": src, "city": city, "station": st, "pollutant": p,
            "coverage": round(float(cov), 4),
            "n_real": int(real.sum()),
            "n_filled": int((g[p].notna() & g[f"{p}_filled"]).sum()),
            "status": "never" if g[p].notna().sum() == 0
                      else ("ok" if cov >= OK_THRESHOLD else "sparse"),
        })

cap = pd.DataFrame(rows)
print(cap.status.value_counts().to_string(), "\n")
print(cap.pivot_table(index=["city", "station"], columns="pollutant",
                      values="coverage", aggfunc="first")
         .reindex(columns=POLLUTANTS)
         .to_string(float_format=lambda v: f"{v:5.2f}", na_rep="  -  "))

status
ok        95
never     31
sparse    12 

pollutant                 pm10  pm25   so2   no2    o3    co
city       station                                          
Banja Luka Banja Luka     0.81  0.80  0.86  0.85  0.85  0.84
Bihac      Bihac          0.77  0.77  0.77  0.68  0.84  0.82
Brod       Brod           0.96  0.94  0.96  0.99  0.96  0.98
Doboj      Doboj          0.61  0.00  0.69  0.52  0.60  0.28
Gacko      Gacko          0.83  0.00  0.94  0.76  0.00  0.00
Livno      Livno          0.76  0.76  0.71  0.84  0.83  0.84
Mostar     Mostar         0.53  0.53  0.56  0.52  0.41  0.58
Prijedor   Prijedor       0.43  0.41  0.65  0.58  0.48  0.55
Sarajevo   Ambasada       0.00  0.99  0.00  0.00  0.00  0.00
           Bjelave        0.95  0.71  0.74  0.89  0.85  0.72
           Hadzici        0.25  0.00  0.52  0.61  0.58  0.61
           Ilidza         0.99  0.99  0.91  0.97  0.00  0.18
           Ilijas         0.92  0.00  0.91  0.93  0.00  0.00
           Isedlo         0.77  0.00 

In [6]:
groups = {
    "threshold_ok": OK_THRESHOLD,
    "definitions": {
        "never":  "instrument absent from source - out of scope, do not impute",
        "sparse": f"present but < {OK_THRESHOLD:.0%} real coverage - use with care",
        "ok":     f">= {OK_THRESHOLD:.0%} real coverage",
    },
    "by_pollutant": {
        p: {s: sorted(cap[(cap.pollutant == p) & (cap.status == s)].station)
            for s in ("ok", "sparse", "never")}
        for p in POLLUTANTS
    },
    "by_station": {
        st: {p: cap[(cap.station == st) & (cap.pollutant == p)].status.iat[0]
             for p in POLLUTANTS}
        for st in sorted(cap.station.unique())
    },
    "coverage": {f"{r.station}|{r.pollutant}": r.coverage
                 for r in cap.itertuples()},
}

with open(os.path.join(SHARED, "station_groups.json"), "w") as f:
    json.dump(groups, f, indent=2, ensure_ascii=False)
cap.to_csv(os.path.join(SHARED, "station_capability.csv"), index=False)

print("NEVER MEASURED - out of scope, not an imputation problem\n")
for p in POLLUTANTS:
    n = groups["by_pollutant"][p]["never"]
    if n:
        print(f"  {p:5s} {len(n):2d}: {', '.join(n)}")
print("\nwrote station_groups.json, station_capability.csv")

NEVER MEASURED - out of scope, not an imputation problem

  pm10   4: Ambasada, Tuzla-BKC, Tuzla-Bukinje, Tuzla-Skver
  pm25   7: Doboj, Gacko, Ilijas, Isedlo, Tuzla-Trnovac, Ugljevik, Vijecnica
  so2    2: Ambasada, Tuzla-Trnovac
  no2    2: Ambasada, Tuzla-Trnovac
  o3     8: Ambasada, Gacko, Ilidza, Ilijas, Tuzla-Trnovac, Ugljevik, Vijecnica, Vogosca
  co     8: Ambasada, Gacko, Ilijas, Isedlo, Otoka, Tuzla-Trnovac, Ugljevik, Vogosca

wrote station_groups.json, station_capability.csv


## 3. Station coordinates - unblocks the GNN

`meta podaci.xlsx` in the FHZ pollutant folder holds the official station
register: latitude, longitude, elevation and station type (urban, industrial,
background). That covers the FHZ network.

It does **not** cover the seven RHZ stations, nor Ambasada (the US embassy
monitor, which is not in the FHZ register). Those eight are filled in from
published city locations and are flagged `approx` in the output - they are good
to roughly a kilometre, which is fine for building a distance-based adjacency
but must not be quoted as station coordinates in the paper. Person B should ask
RHZ for the exact figures.

In [7]:
# The official FHZ station register. Read from the source workbook when it is
# present; otherwise the copy embedded below is used, so this notebook runs
# anywhere with only bih_hourly_clean.csv to download.
REGISTER_CSV = """code,name,lat,lon,elev,type
BA0001G,Ivan Sedlo,43.778,18.02,969,Regionalna pozadinska
BA0029A,Sarajevo Bjelave,43.867,18.423,635,Urbano pozadinska
BA0031A,Tuzla Skver,44.54,18.673,234,Urbana/saobraćajna
BA0032A,Tuzla BKC,44.534,18.661,231,Urbana
BA0036A,Zenica Brist,44.202,17.9,341,Urbano pozadinska
BA0037A,Zenica Centar,44.198,17.912,335,Urbana
BA0038A,Zenica Tetovo,44.225,17.89,337,Industrijska
BA0039A,Zenica Radakovo,44.195,17.931,340,Urbana/saobraćajna
BA0040A,Jajce Harmani,44.343,17.267,401,Urbano pozadinska
BA0041A,Goražde Rasadnik,43.661,18.977,361,Urbano pozadinska
BA0042A,Sarajevo Otoka,43.848,18.363,512,Urbana/saobraćajna
BA0043A,Sarajevo Ilidza,43.83,18.31,509,Urbana
BA0044A,Tuzla Bukinje,44.523,18.6,214,Industrijska
BA0045A,Lukavac Centar,44.533,18.534,187,Urbana
BA0046A,Živinice Centar,44.454,18.648,214,Urbana
BA0049A,Sarajevo Vijecnica,43.859,18.434,554,Urbana/saobraćajna
BA0050A,Sarajevo Ilijas,43.96,18.269,459,Urbano pozadinska
BA0051A,Zenica Vranduk,44.289,17.907,359,Ruralno pozadinska
BA0054A,Maglaj Centar,44.544,18.098,175,Urbana
BA0055A,Visoko Centar,43.994,18.175,425,Urbana
BA0058A,Bihac Nova Četvrt,44.807,15.866,244,Urbano pozadinska
BA0057A,Livno Centar,43.822,17.001,806,Urbano pozadinska
BA0056A,Tesanj Vatrogasno,44.619,17.991,240,Urbano pozadinska
BA0060A,Hadžići,43.823,18.201,557,Urbana
BA0061A,Vogošća Centar,43.9,18.342,496,Urbana
BA0062A,Travnik Centar,44.225,17.667,507,Urbana
BA0068A,Tuzla Trnovac,44.542,18.689,299,Urbano pozadinska
BA0066A,Kakanj Centar,44.124,18.115,388,Urbana
BA0067A,Mostar Bijeli Brijeg,43.348,17.794,97,Urbano pozadinska
BA0069A,Vareš Centar,44.157,18.325,821,Urbana
BA0070A,Kakanj Općina,44.123,18.115,388,Urbana
BA0071A,Sarajevo Saraj Polje,43.837,18.342,Urbana,32
BA0072A,Mostar Kampus,43.354,17.809,Urbana pozadinska,33"""

import io
if os.path.exists(META):
    meta = pd.read_excel(META, header=None, skiprows=3).iloc[:, 1:7]
    meta.columns = ["code", "name", "lat", "lon", "elev", "type"]
    print("register: read from", os.path.basename(META))
else:
    meta = pd.read_csv(io.StringIO(REGISTER_CSV))
    print("register: using the embedded copy (source workbook not present)")

meta = meta.dropna(subset=["name"])
meta["name"] = meta["name"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# The last two rows of the register have elevation and type swapped.
swapped = pd.to_numeric(meta["elev"], errors="coerce").isna()
meta.loc[swapped, ["elev", "type"]] = meta.loc[swapped, ["type", "elev"]].values
meta["elev"] = pd.to_numeric(meta["elev"], errors="coerce")

# our station name -> name in the official register
FHZ_MATCH = {
    "Bihac": "Bihac Nova Četvrt",   "Livno": "Livno Centar",
    # The register lists two Mostar stations. Ours is Bijeli Brijeg: the 2025
    # raw sheets label the same single column "Mostar Bijeli brijeg" (SO2),
    # "Mostar - Bijeli Brijeg" (O3) and "Mostar Bijeli Brijeg" (CO), while the
    # PM and NO2 sheets shorten it to "Mostar". Kampus is not in the data.
    "Mostar": "Mostar Bijeli Brijeg",
    "Bjelave": "Sarajevo Bjelave",  "Hadzici": "Hadžići",
    "Ilidza": "Sarajevo Ilidza",    "Ilijas": "Sarajevo Ilijas",
    "Isedlo": "Ivan Sedlo",         "Otoka": "Sarajevo Otoka",
    "Vijecnica": "Sarajevo Vijecnica", "Vogosca": "Vogošća Centar",
    "Tuzla-BKC": "Tuzla BKC",       "Tuzla-Bukinje": "Tuzla Bukinje",
    "Tuzla-Skver": "Tuzla Skver",   "Tuzla-Trnovac": "Tuzla Trnovac",
}

# Not in the FHZ register.
#
# Ambasada is the US embassy monitor (EPA/AirNow programme). The embassy's
# published location is 43 51 23.89 N, 18 24 01.5 E, so this one is a real
# address, not a guess.
#
# RHMZ RS metadata (lat, lon, type), from RHMZ RS station descriptions -
# supersedes the old city-centre APPROX guesses for these seven, and adds
# Bijeljina (two measuring points), which wasn't covered at all before.
# DMS coordinates for Bijeljina converted to decimal degrees.
RS_META = {
    "Banja Luka": (44.79804,  17.20496,  "Urbana pozadinska"),
    "Prijedor":   (44.971993, 16.712734, "Urbana pozadinska"),
    "Trebinje":   (42.71408,  18.33418,  "Urbana pozadinska"),
    "Doboj":      (44.7262,   18.0891,   "Urbana pozadinska"),
    "Brod":       (45.135565, 17.984798, "Urbana industrijska"),
    "Ugljevik":   (44.684038, 18.969576, "Industrijska"),
    "Gacko":      (43.16458,  18.535791, "Urbana pozadinska"),
    "Bijeljina Centar":  (44.756786, 19.218481, "Urbana"),
    "Bijeljina Toplana": (44.761633, 19.205911, "Urbana industrijska"),
}
LOOKED_UP = {
    "Ambasada": (43.8566, 18.4004, 540),
}

recs = []
for (src, city, st) in df.groupby(["source", "city", "station"]).groups:
    if st in FHZ_MATCH:
        m = meta[meta.name == FHZ_MATCH[st]]
        if len(m) != 1:
            raise SystemExit(f"register lookup failed for {st} -> {FHZ_MATCH[st]}")
        m = m.iloc[0]
        recs.append(dict(source=src, city=city, station=st, lat=float(m.lat),
                         lon=float(m.lon), elev_m=float(m.elev),
                         station_type=str(m.type).strip(), code=m.code,
                         precision="official"))
    elif st in RS_META:
        lat, lon, stype = RS_META[st]
        recs.append(dict(source=src, city=city, station=st, lat=lat, lon=lon,
                         elev_m=np.nan, station_type=stype, code="",
                         precision="official"))
    elif st in LOOKED_UP:
        lat, lon, elev = LOOKED_UP[st]
        recs.append(dict(source=src, city=city, station=st, lat=lat, lon=lon,
                         elev_m=elev, station_type="unknown", code="",
                         precision="published"))
    else:
        recs.append(dict(source=src, city=city, station=st, lat=np.nan,
                         lon=np.nan, elev_m=np.nan, station_type="unknown",
                         code="", precision="MISSING"))

coords = pd.DataFrame(recs).sort_values(["source", "city", "station"])
coords.to_csv(os.path.join(SHARED, "station_coords.csv"), index=False)

print(coords.to_string(index=False))
print("\n", coords.precision.value_counts().to_string(), sep="")
assert (coords.precision != "MISSING").all(), "some station has no coordinates"

register: using the embedded copy (source workbook not present)
source       city       station       lat       lon  elev_m          station_type    code precision
   fhz      Bihac         Bihac 44.807000 15.866000   244.0     Urbano pozadinska BA0058A  official
   fhz      Livno         Livno 43.822000 17.001000   806.0     Urbano pozadinska BA0057A  official
   fhz     Mostar        Mostar 43.348000 17.794000    97.0     Urbano pozadinska BA0067A  official
   fhz   Sarajevo      Ambasada 43.856600 18.400400   540.0               unknown         published
   fhz   Sarajevo       Bjelave 43.867000 18.423000   635.0     Urbano pozadinska BA0029A  official
   fhz   Sarajevo       Hadzici 43.823000 18.201000   557.0                Urbana BA0060A  official
   fhz   Sarajevo        Ilidza 43.830000 18.310000   509.0                Urbana BA0043A  official
   fhz   Sarajevo        Ilijas 43.960000 18.269000   459.0     Urbano pozadinska BA0050A  official
   fhz   Sarajevo        Isedlo 43.7

In [8]:
# Great-circle distance matrix, ready for a GNN adjacency.
R = 6371.0
la = np.radians(coords.lat.to_numpy()); lo = np.radians(coords.lon.to_numpy())
dla = la[:, None] - la[None, :]; dlo = lo[:, None] - lo[None, :]
a = np.sin(dla / 2) ** 2 + np.cos(la)[:, None] * np.cos(la)[None, :] * np.sin(dlo / 2) ** 2
D = 2 * R * np.arcsin(np.sqrt(a))

dist = pd.DataFrame(D, index=coords.station, columns=coords.station).round(2)
dist.to_csv(os.path.join(SHARED, "station_distance_km.csv"))

print("pairwise distance, km - min "
      f"{D[D > 0].min():.2f}, median {np.median(D[D > 0]):.1f}, max {D.max():.1f}")
print("\nclosest pairs:")
iu = np.triu_indices_from(D, k=1)
for i in np.argsort(D[iu])[:6]:
    r, c = iu[0][i], iu[1][i]
    print(f"  {coords.station.iat[r]:>14s} - {coords.station.iat[c]:<14s} {D[r, c]:6.2f} km")

pairwise distance, km - min 1.16, median 100.3, max 305.7

closest pairs:
       Tuzla-BKC - Tuzla-Skver      1.16 km
         Bjelave - Vijecnica        1.25 km
     Tuzla-Skver - Tuzla-Trnovac    1.29 km
        Ambasada - Bjelave          2.15 km
       Tuzla-BKC - Tuzla-Trnovac    2.39 km
        Ambasada - Vijecnica        2.71 km


## 4. Freeze the evaluation windows

The strongest form of "same split": rather than three notebooks each deciding
which 2024 windows are scoreable, the eligible windows are enumerated **once**
and written out. Every track scores the same origins on the same series.

Eligibility is decided on the **target side only** - at least 12 of the 24 hours
must be real and unfilled, and there must be some history in the preceding week.
Deliberately nothing about context length, since Chronos wants 512 hours and a
GNN or xLSTM may use far less; putting a context rule here would impose one
model's appetite on the others.

In [9]:
MIN_REAL_TARGET = SPLIT["masking"]["min_real_target_hours"]

full = pd.date_range(df.datetime.min(), df.datetime.max(), freq="h")
panel = {}          # (station, pollutant) -> DataFrame(y, filled) on `full`

for (src, city, st), g in df.groupby(["source", "city", "station"], sort=True):
    g = g.drop_duplicates("datetime").set_index("datetime").reindex(full)
    for p in POLLUTANTS:
        if groups["by_station"][st][p] == "never":
            continue
        panel[(st, p)] = pd.DataFrame(
            {"y": g[p].to_numpy(float),
             "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)}, index=full)

origins = full[(full.year == 2024) & (full.hour == 0)]
pos = {t: i for i, t in enumerate(full)}

wins = []
for (st, p), s in panel.items():
    y, fl = s.y.to_numpy(), s.filled.to_numpy()
    real = ~np.isnan(y) & ~fl
    for o_ts in origins:
        o = pos[o_ts]
        if o + HORIZON > len(y):
            continue
        n_real = int(real[o:o + HORIZON].sum())
        if n_real < MIN_REAL_TARGET:
            continue
        if not np.isfinite(y[max(0, o - 168):o]).any():
            continue
        wins.append({"station": st, "pollutant": p, "origin": o_ts,
                     "n_real_target": n_real})

ev = pd.DataFrame(wins)
ev = ev.merge(coords[["station", "city", "source"]], on="station", how="left")
ev = ev[["source", "city", "station", "pollutant", "origin", "n_real_target"]]
ev.to_csv(os.path.join(SHARED, "eval_windows.csv"), index=False)

print(f"{len(ev):,} evaluation windows across "
      f"{ev.groupby(['station', 'pollutant']).ngroups} station-pollutant series\n")
print(ev.pivot_table(index="pollutant", values="origin", aggfunc="count")
        .rename(columns={"origin": "windows"}).to_string())

/tmp/ipykernel_553/3113324929.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)}, index=full)
/tmp/ipykernel_553/3113324929.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)}, index=full)
/tmp/ipykernel_553/3113324929.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set

31,688 evaluation windows across 98 station-pollutant series

           windows
pollutant         
co            4105
no2           7055
o3            4390
pm10          5001
pm25          4151
so2           6986


## 5. The shared module every track imports

In [10]:
SHARED_PY = '''"""Shared loading, masking and metrics for the BIH air-quality project.

Import this from the GNN, xLSTM and Chronos notebooks so all three use one
definition of the split, the mask and the metrics.

    import sys; sys.path.append(SHARED_DIR)
    import bih_shared as bs

    split  = bs.load_split(SHARED_DIR)
    panel  = bs.load_panel(CLEAN, SHARED_DIR)
    wins   = bs.load_windows(SHARED_DIR)
    truth  = bs.target(panel, station, pollutant, origin)   # y, mask
    scores = bs.score(pred, truth.y, truth.mask, scale)
"""
import json, os
import numpy as np
import pandas as pd

POLLUTANTS = ["pm10", "pm25", "so2", "no2", "o3", "co"]
HORIZON = 24


def load_split(shared_dir):
    with open(os.path.join(shared_dir, "split.json")) as f:
        return json.load(f)


def load_groups(shared_dir):
    with open(os.path.join(shared_dir, "station_groups.json")) as f:
        return json.load(f)


def load_windows(shared_dir):
    return pd.read_csv(os.path.join(shared_dir, "eval_windows.csv"),
                       parse_dates=["origin"])


def load_coords(shared_dir):
    return pd.read_csv(os.path.join(shared_dir, "station_coords.csv"))


def load_panel(clean_csv, shared_dir):
    """{(station, pollutant): DataFrame(y, filled)} on a gap-free hourly index.

    Reindexing to a complete hourly range matters: a missing hour has to exist
    as NaN, or every model silently treats the next reading as one hour later
    than it is.
    """
    groups = load_groups(shared_dir)
    cols = (["source", "city", "station", "datetime"] + POLLUTANTS
            + [f"{p}_filled" for p in POLLUTANTS])
    df = pd.read_csv(clean_csv, usecols=cols, parse_dates=["datetime"])
    full = pd.date_range(df.datetime.min(), df.datetime.max(), freq="h")
    panel = {}
    for st, g in df.groupby("station", sort=True):
        g = g.drop_duplicates("datetime").set_index("datetime").reindex(full)
        for p in POLLUTANTS:
            if groups["by_station"].get(st, {}).get(p) == "never":
                continue
            panel[(st, p)] = pd.DataFrame(
                {"y": g[p].to_numpy(float),
                 "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
                index=full)
    return panel


def target(panel, station, pollutant, origin, horizon=HORIZON):
    """The 24 hours to predict, and the mask of what may be scored.

    mask is False wherever the value is missing or was interpolated during
    dataset build - those points are not ground truth.
    """
    s = panel[(station, pollutant)]
    i = s.index.get_loc(pd.Timestamp(origin))
    y = s.y.to_numpy()[i:i + horizon]
    fl = s.filled.to_numpy()[i:i + horizon]
    return y, (~np.isnan(y) & ~fl)


def context(panel, station, pollutant, origin, length):
    """The `length` hours before the origin. Filled values are allowed here."""
    s = panel[(station, pollutant)]
    i = s.index.get_loc(pd.Timestamp(origin))
    return s.y.to_numpy()[max(0, i - length):i]


def mase_scale(ctx, season=24):
    """In-context seasonal-naive MAE - the MASE denominator."""
    if len(ctx) <= season:
        return np.nan
    d = np.abs(ctx[season:] - ctx[:-season])
    return float(np.nanmean(d)) if np.isfinite(d).any() else np.nan


def score(pred, y, mask, scale=np.nan):
    """MAE, RMSE and MASE over masked points only."""
    err = np.where(mask, np.asarray(pred, float) - y, np.nan)
    mae = float(np.nanmean(np.abs(err)))
    return {"mae": mae,
            "rmse": float(np.sqrt(np.nanmean(err ** 2))),
            "mase": mae / scale if np.isfinite(scale) and scale > 0 else np.nan,
            "n": int(mask.sum())}


def aggregate(rows):
    """Per-pollutant means. Never average MAE/RMSE across pollutants - CO is
    mg/m3 and the rest ug/m3. MASE is the only cross-pollutant summary."""
    r = pd.DataFrame(rows)
    return r.groupby(["model", "pollutant"])[["mae", "rmse", "mase"]].mean()
'''

with open(os.path.join(SHARED, "bih_shared.py"), "w") as f:
    f.write(SHARED_PY)
print("wrote bih_shared.py")

wrote bih_shared.py


## 6. The shared baselines

Three, all computed from information available at the forecast origin:

- **persistence_t24** - the next 24 hours repeat the previous 24. On hourly
  pollution this is strong, because the daily heating-and-traffic cycle
  dominates. This is the bar.
- **persistence_last** - the whole day equals the last observed value. A floor,
  not a serious competitor.
- **ratio_pm25** - PM2.5 predicted as a season-specific ratio times the PM10
  forecast. The ratio is fitted **on train only** (2021-2023), never on 2024,
  and the PM10 it multiplies is itself a t-24 persistence forecast, so the
  baseline never sees the future.

That last one is what stops a "PM2.5 model" that has quietly learned only
`pm25 = 0.75 x pm10` from looking impressive.

In [11]:
# Season ratios fitted on TRAIN ONLY - using 2024 here would leak the answer.
tr = df[df.datetime <= TRAIN_END]
both = tr[(tr.pm10 > 0) & tr.pm25.notna() & ~tr.pm10_filled & ~tr.pm25_filled]
RATIO = (both.pm25 / both.pm10).groupby(both.season).median().to_dict()

print("PM2.5 / PM10 median ratio, fitted on 2021-2023 only:")
for s in ["winter", "spring", "summer", "autumn"]:
    n = int((both.season == s).sum())
    print(f"  {s:7s} {RATIO[s]:.3f}   (n = {n:,})")
with open(os.path.join(SHARED, "pm25_pm10_ratio.json"), "w") as f:
    json.dump({"fitted_on": "2021-2023 train only", "median_ratio": RATIO}, f, indent=2)

PM2.5 / PM10 median ratio, fitted on 2021-2023 only:
  winter  0.894   (n = 40,688)
  spring  0.750   (n = 41,529)
  summer  0.643   (n = 46,991)
  autumn  0.736   (n = 48,119)


In [12]:
def season_of(ts):
    m = ts.month
    return ("winter" if m in (12, 1, 2) else "spring" if m in (3, 4, 5)
            else "summer" if m in (6, 7, 8) else "autumn")


def run_baselines(ev, panel):
    rows = []
    for w in ev.itertuples():
        key = (w.station, w.pollutant)
        s = panel[key]
        i = s.index.get_loc(w.origin)
        y = s.y.to_numpy()[i:i + HORIZON]
        fl = s.filled.to_numpy()[i:i + HORIZON]
        mask = ~np.isnan(y) & ~fl
        ctx = s.y.to_numpy()[max(0, i - 512):i]

        d = np.abs(ctx[24:] - ctx[:-24]) if len(ctx) > 24 else np.array([np.nan])
        scale = np.nanmean(d) if np.isfinite(d).any() else np.nan

        prev = s.y.to_numpy()[i - 24:i]                       # yesterday, same hours
        p24 = np.where(np.isnan(prev), np.nanmean(ctx), prev)
        ok = np.flatnonzero(~np.isnan(ctx))
        plast = np.repeat(ctx[ok[-1]] if len(ok) else np.nan, HORIZON)

        preds = {"persistence_t24": p24, "persistence_last": plast}

        if w.pollutant == "pm25" and (w.station, "pm10") in panel:
            s10 = panel[(w.station, "pm10")]
            prev10 = s10.y.to_numpy()[i - 24:i]
            if np.isfinite(prev10).any():
                r = RATIO[season_of(w.origin)]
                preds["ratio_pm25"] = np.where(
                    np.isnan(prev10), np.nanmean(prev10), prev10) * r

        for name, pred in preds.items():
            err = np.where(mask, pred - y, np.nan)
            mae = np.nanmean(np.abs(err))
            rows.append({
                "model": name, "station": w.station, "city": w.city,
                "source": w.source, "pollutant": w.pollutant, "origin": w.origin,
                "season": season_of(w.origin), "mae": mae,
                "rmse": np.sqrt(np.nanmean(err ** 2)),
                "mase": mae / scale if np.isfinite(scale) and scale > 0 else np.nan,
                "n": int(mask.sum()),
            })
    return pd.DataFrame(rows)


base = run_baselines(ev, panel)
base.to_csv(os.path.join(SHARED, "baseline_windows.csv"), index=False)
print(len(base), "baseline scores")

66606 baseline scores


In [13]:
per_pol = base.groupby(["pollutant", "model"])[["mae", "rmse", "mase"]].mean()
per_pol.to_csv(os.path.join(SHARED, "baseline_metrics.csv"))

print("THE BAR - per pollutant, native units, real unfilled hours only\n")
print(per_pol.to_string(float_format=lambda v: f"{v:8.3f}"))

print("\n\nMASE by model (cross-pollutant summary; 1.0 = the in-context naive)\n")
print(base.groupby("model").mase.mean().sort_values()
          .to_string(float_format=lambda v: f"{v:.3f}"))

print("\n\npersistence_t24 MASE by season\n")
print(base[base.model == "persistence_t24"]
        .pivot_table(index="season", columns="pollutant", values="mase")
        .reindex(["winter", "spring", "summer", "autumn"])
        .to_string(float_format=lambda v: f"{v:6.3f}"))

THE BAR - per pollutant, native units, real unfilled hours only

                                mae     rmse     mase
pollutant model                                      
co        persistence_last    0.232    0.272    1.394
          persistence_t24     0.171    0.226    1.053
no2       persistence_last   10.407   12.239    1.512
          persistence_t24     7.446    9.796    1.040
o3        persistence_last   22.061   27.710    1.403
          persistence_t24    16.922   21.030    1.050
pm10      persistence_last   18.395   21.719    1.162
          persistence_t24    16.746   21.114    1.077
pm25      persistence_last   16.054   18.766    1.203
          persistence_t24    13.282   16.846    1.052
          ratio_pm25         13.515   16.888    1.213
so2       persistence_last    8.323   12.359    1.043
          persistence_t24     9.221   15.229    1.159


MASE by model (cross-pollutant summary; 1.0 = the in-context naive)

model
persistence_t24    1.077
ratio_pm25         1.21

## What was written

In [14]:
print("dataset/shared/\n")
for f in sorted(os.listdir(SHARED)):
    print(f"  {f:28s} {os.path.getsize(os.path.join(SHARED, f)) / 1e3:9.1f} KB")

print(f"""

Published at
  https://drive.google.com/drive/folders/{SHARED_FOLDER_ID}

The GNN, xLSTM and Chronos notebooks do not re-run this notebook. They start
with these five lines, which need nothing pasted in:

    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown; gdown.download_folder(id="{SHARED_FOLDER_ID}", output="shared", quiet=True)
    sys.path.append("shared")
    import bih_shared as bs

then:

    panel   = bs.load_panel(CLEAN, "shared")   # gap-free hourly series
    wins    = bs.load_windows("shared")        # the frozen 2024 windows
    y, mask = bs.target(panel, station, pollutant, origin)
    bs.score(pred, y, mask, scale)             # MAE / RMSE / MASE

Train on 2021-2023 only. Score only where mask is True.
If any artifact is regenerated, re-upload that one file to the folder - do not
keep a private copy, or the three sets of results stop being comparable.
""")

dataset/shared/

  baseline_metrics.csv               1.0 KB
  baseline_windows.csv            7772.6 KB
  bih_shared.py                      3.9 KB
  eval_windows.csv                1200.2 KB
  pm25_pm10_ratio.json               0.2 KB
  split.json                         1.0 KB
  station_capability.csv             6.0 KB
  station_coords.csv                 1.7 KB
  station_distance_km.csv            3.7 KB
  station_groups.json               10.5 KB


Published at
  https://drive.google.com/drive/folders/1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE

The GNN, xLSTM and Chronos notebooks do not re-run this notebook. They start
with these five lines, which need nothing pasted in:

    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown; gdown.download_folder(id="1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE", output="shared", quiet=True)
    sys.path.append("shared")
    import bih_shared as bs

then:

    panel   = bs.load_panel(CLE

## 7. Package it for Drive

Zips `dataset/shared/` - the ten data artifacts - and downloads it.

Two folders go to Drive, and they are kept apart on purpose:

- **`dataset/shared/`** - the artifacts. This is the folder the model
  notebooks download with `gdown.download_folder`.
- **`code/shared/`** - `shared_setup.ipynb` and `chronos_forecast.ipynb`.
  Notebooks for people to open, never downloaded by code.


In [15]:
import shutil

# dataset/shared holds DATA ARTIFACTS ONLY. This notebook lives in code/shared
# and is published separately - mixing the two is what makes it unclear which
# Drive folder is which.
archive = shutil.make_archive("bih_shared", "zip", SHARED)
print(f"{os.path.abspath(archive)}  ({os.path.getsize(archive) / 1e6:.1f} MB)\n")
for f in sorted(os.listdir(SHARED)):
    print(f"   {f}")

try:
    from google.colab import files
    files.download(archive)          # lands in your browser's Downloads folder
except ImportError:
    print("\nnot in Colab - the zip is at", os.path.abspath(archive))

/content/bih_shared.zip  (2.2 MB)

   baseline_metrics.csv
   baseline_windows.csv
   bih_shared.py
   eval_windows.csv
   pm25_pm10_ratio.json
   split.json
   station_capability.csv
   station_coords.csv
   station_distance_km.csv
   station_groups.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>